# Module 11 lab · High dimensions and embeddings

The code from every lesson of [this module](https://aicourses.dev/mathematics/high-dimensions), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 11.2 Distances concentrate

In many dimensions the nearest and the farthest random point are almost the same distance away. You see it in numbers, and it explains why plain nearest-neighbour search gets harder.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/distances-concentrate)

### In NumPy

The function below measures the nearest distance over the farthest for one query among the points. It leaves the query’s own distance (zero) out by setting it to `nan`, “not a number”, which `np.nanmin` and `np.nanmax` skip. The loop draws 1,000 points in a unit cube and, for comparison, with bell-shaped coordinates, and averages over the first 20 points as queries.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def ratio(P, q):                       # nearest / farthest from point q
    dist = np.linalg.norm(P - P[q], axis=1)
    dist[q] = np.nan                   # leave out the distance to itself
    return np.nanmin(dist) / np.nanmax(dist)

for d in (2, 10, 100, 1000):
    cube = rng.random((1000, d))       # 1,000 points in the unit cube
    gauss = rng.normal(size=(1000, d)) # or with bell-shaped coordinates
    print(d, np.mean([ratio(cube, q) for q in range(20)]).round(3),
          np.mean([ratio(gauss, q) for q in range(20)]).round(3))

*The lesson shows:*

```text
2 0.018 0.013
10 0.245 0.245
100 0.691 0.677
1000 0.892 0.881
```

### Why distances bunch up

The zeros-and-ones points make the reason checkable. The loop below draws them and measures the squared distances from point 0.

In [ ]:
for d in (2, 10, 100, 1000):
    P = rng.integers(0, 2, size=(1000, d))      # every coordinate 0 or 1
    sq = ((P[1:] - P[0]) ** 2).sum(axis=1)      # squared distances to point 0
    print(d, sq.mean().round(1), sq.std().round(2), d / 2,
          (np.sqrt(d) / 2).round(2))

*The lesson shows:*

```text
2 1.0 0.73 1.0 0.71
10 5.0 1.62 5.0 1.58
100 50.1 4.97 50.0 5.0
1000 500.6 16.37 500.0 15.81
```

### Real vectors are not random points

Everything so far is for random points whose coordinates do not depend on each other. Real data is different: the Wikipedia article on the curse of dimensionality adds, citing a survey, that when attributes are correlated the data can become easier to tell apart, with more contrast between near and far. The course’s word vectors are real data, with 10,000 words of 100 numbers each. This block loads them and measures the same ratio from 20 random query words, among all the other words.

In [ ]:
import urllib.request

DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
V = np.load("glove.npz")["vectors"].astype(float)   # 10,000 words by 100
queries = rng.choice(10_000, 20, replace=False)
r = [ratio(V, q) for q in queries]
print(np.mean(r).round(3), np.min(r).round(3), np.max(r).round(3))

*The lesson shows:*

```text
0.243 0.087 0.34
```

## 11.3 Almost everything is nearly perpendicular

Two random directions are nearly perpendicular, with a spread you can compute from the number of dimensions. It is the reason attention scores are scaled in Module 12.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/nearly-perpendicular)

### By hand: lists of plus and minus ones

Take the simplest random vectors: 4 entries, each +1 or −1. There are 16 such lists. The dot product of two of them counts the places they agree minus the places they differ, so it is 4, 2, 0, −2 or −4. Over all 16 × 16 = 256 pairs, the counts are 16, 64, 96, 64 and 16 for those five values, which gives a spread of 2, the square root of 4. (Square each value, weight it by its count and average: $(16 \times 16 + 64 \times 4 + 96 \times 0 + 64 \times 4 + 16 \times 16) / 256 = 4$, and the square root of 4 is 2.) Every such list has length exactly 2, so the cosine is the dot product over 4, with a spread of 0.5, which is one over the square root of 4. The code confirms it.

In [ ]:
import numpy as np
from itertools import product

rng = np.random.default_rng(0)
signs = np.array(list(product([-1, 1], repeat=4)))  # all 16 lists of 4 signs
dots = (signs[:, None, :] * signs[None, :, :]).sum(axis=2).ravel()
print(np.unique(dots, return_counts=True))
print(dots.std(), (dots / 4).std())

*The lesson shows:*

```text
(array([-4, -2,  0,  2,  4]), array([16, 64, 96, 64, 16]))
2.0 0.5
```

### Why one over the square root

Worked for $d = 100$: the dot product has spread 10, each length is about 10, and the cosine has spread $10 / (10 \times 10) = 0.1$. For $d = 4$ the same steps give 2 and $0.5$, the numbers you counted by hand. The steps are an approximation (the lengths vary a little), and the code shows how good it is.

In [ ]:
for d in (2, 10, 100, 512):
    a = rng.normal(size=(10_000, d))          # 10,000 random vectors
    b = rng.normal(size=(10_000, d))          # and 10,000 partners
    dot = (a * b).sum(axis=1)
    cos = dot / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    print(d, cos.std().round(4), (1 / np.sqrt(d)).round(4),
          dot.std().round(2), np.sqrt(d).round(2))

*The lesson shows:*

```text
2 0.7046 0.7071 1.42 1.41
10 0.3153 0.3162 3.16 3.16
100 0.0991 0.1 9.93 10.0
512 0.0443 0.0442 22.68 22.63
```

### The angle itself

A cosine near 0 means an angle near 90 degrees. Here is the angle of 10,000 random pairs in 100 directions, in degrees, with the small-angle prediction: a cosine spread of $1/\sqrt{d}$ is an angle spread of $(180/\pi)/\sqrt{d}$ degrees.

In [ ]:
d = 100
a, b = rng.normal(size=(10_000, d)), rng.normal(size=(10_000, d))
lengths = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
angle = np.degrees(np.arccos((a * b).sum(axis=1) / lengths))
print(angle.mean().round(1), angle.std().round(2),
      np.degrees(1 / np.sqrt(d)).round(2))
print(np.percentile(angle, [5, 95]).round(1))

*The lesson shows:*

```text
90.1 5.77 5.73
[80.5 99.6]
```

### Where this lives in AI

Attention gives each query a score for each key and turns the scores into weights with a softmax (lesson 09.3). The block below does it for 8 keys. Queries and keys have 512 independent entries of spread 1, as in the footnote’s illustration, and each query gets 8 scores. It reports the spread of the scores and the average of the largest weight, before and after dividing the scores by $\sqrt{512} = 22.6$.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max(axis=1, keepdims=True))     # subtract the max (09.7)
    return e / e.sum(axis=1, keepdims=True)

q = rng.normal(size=(1000, 512))
k = rng.normal(size=(1000, 8, 512))                  # 8 keys for each query
scores = (k * q[:, None, :]).sum(axis=2)             # 8 dot products each
for name, s in (("raw", scores), ("scaled", scores / np.sqrt(512))):
    p = softmax(s)
    print(name, s.std().round(2), p.max(axis=1).mean().round(3),
          (p.max(axis=1) > 0.99).mean().round(3))

*The lesson shows:*

```text
raw 22.87 0.958 0.742
scaled 1.01 0.364 0.0
```

## 11.4 Dot product or cosine?

When length carries meaning you want the dot product, and when it is noise you want the cosine. You learn which to use, what normalising throws away, and what the course’s word vectors show about length.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/dot-or-cosine)

### Is length a signal in this table?

A common story says a word vector’s length carries something about the word, such as how often it is used. Check it on the course’s table, rather than trust it. The block loads the 10,000 words, which come in order of how often they are used in the text the vectors were trained on, and prints the ends of the length ranking.

In [ ]:
import numpy as np
import urllib.request

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
d = np.load("glove.npz")
words = [str(w) for w in d["words"]]
V = d["vectors"].astype(float)                 # 10,000 words, 100 numbers each
length = np.linalg.norm(V, axis=1)
order = np.argsort(length)
print([(words[i], round(float(length[i]), 2)) for i in order[:3]])
print([(words[i], round(float(length[i]), 2)) for i in order[::-1][:3]])

*The lesson shows:*

```text
[('likewise', 2.51), ('meantime', 3.11), ('furthermore', 3.15)]
[('non-families', 12.31), ('householder', 10.28), ('coxnet', 10.27)]
```

These are the ends from the opening. Now the relation to frequency. A **rank correlation** replaces each number by its place in line (smallest first) and takes the ordinary correlation of lesson 08.2 of those places, so it measures whether two things rise together without assuming a straight line. Here the places of the lengths go against the places of the words in frequency order (0 is the most frequent).

In [ ]:
rank = np.arange(10_000)                       # 0 = the most frequent word
def ranks(x):
    return np.argsort(np.argsort(x))           # each value's place in line

print(np.corrcoef(ranks(length), ranks(rank))[0, 1].round(3))
for lo, hi in [(0, 100), (100, 1000), (1000, 5000), (5000, 10000)]:
    print(lo + 1, hi, length[lo:hi].mean().round(2))

*The lesson shows:*

```text
-0.255
1 100 6.19
101 1000 5.85
1001 5000 5.56
5001 10000 5.37
```

### What the dot product does to a search

If length is mostly noise here, ranking by dot product lets it leak into the answers. Take 300 random query words and, for each, the 10 nearest words by dot product and by cosine. The block counts how many of the 10 the two methods share, and how often a single word turns up in a query’s top 10.

In [ ]:
U = V / length[:, None]                        # every row scaled to length 1
queries = rng.choice(10_000, 300, replace=False)
same, hits_dot, hits_cos = [], np.zeros(10_000), np.zeros(10_000)
for q in queries:
    by_dot, by_cos = V @ V[q], U @ U[q]
    by_dot[q] = by_cos[q] = -1e9               # leave the word itself out
    top_dot, top_cos = np.argsort(-by_dot)[:10], np.argsort(-by_cos)[:10]
    same.append(len(set(top_dot) & set(top_cos)) / 10)
    hits_dot[top_dot] += 1                     # how often a word is in a top 10
    hits_cos[top_cos] += 1
print(np.mean(same).round(3), int(hits_dot.max()), int(hits_cos.max()))
print([(words[i], int(hits_dot[i])) for i in np.argsort(-hits_dot)[:4]])

*The lesson shows:*

```text
0.606 13 4
[('we', 13), ('i', 12), ('you', 11), ('to', 10)]
```

## 11.5 Word vectors: neighbours, analogies and their limits

Real word vectors have neighbours, a few analogies that work and others that fail. This lesson shows what the critics measured, so you can tell the evidence from the story.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/word-vectors)

### Neighbours and the text behind them

The course’s table is the 10,000 most frequent words of the 6B GloVe vectors, 100 numbers each. GloVe (Pennington, Socher and Manning, 2014) fits a vector to every word from counts of how often words occur near each other in a large text. The 6B set was trained on Wikipedia as of 2014 together with Gigaword 5, a collection of newswire stories: about 6 billion tokens, words and punctuation marks, in all (the GloVe project’s description; the paper gives the same size, and the Linguistic Data Consortium describes Gigaword 5 as an archive of newswire text). So the vectors come from encyclopedia and news text, and “house” sits beside “senate” because the two occur in the same places there.

In [ ]:
import numpy as np
import urllib.request

DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
d = np.load("glove.npz")
words = [str(w) for w in d["words"]]
V = d["vectors"].astype(float)
index = {w: i for i, w in enumerate(words)}
U = V / np.linalg.norm(V, axis=1, keepdims=True)   # every row of length 1
print(V.shape, words[:6])

*The lesson shows:*

```text
(10000, 100) ['the', ',', '.', 'of', 'to', 'and']
```

The table has 10,000 words, listed from the most to the least frequent, and punctuation counts as words. `U` holds the same vectors scaled to length 1, so a dot product with a row of `U` is a cosine (lesson 11.4). The function below returns the words with the highest cosines to a given word, leaving the word itself out.

In [ ]:
def nearest(s, k=5):                           # the k words with the top scores
    return [(words[i], round(float(s[i]), 3)) for i in np.argsort(-s)[:k]]

def neighbours(word, k=5):
    s = U @ U[index[word]]                     # cosine with every word
    s[index[word]] = -9                        # leave the word itself out
    return nearest(s, k)

for w in ["car", "money", "house", "happy"]:
    print(w, neighbours(w))

*The lesson shows:*

```text
car [('vehicle', 0.863), ('truck', 0.86), ('cars', 0.837), ('driver', 0.819), ('driving', 0.781)]
money [('funds', 0.851), ('cash', 0.848), ('fund', 0.759), ('paying', 0.742), ('pay', 0.741)]
house [('office', 0.758), ('senate', 0.72), ('room', 0.715), ('houses', 0.689), ('capitol', 0.685)]
happy [("'m", 0.841), ('feel', 0.813), ("'re", 0.805), ('i', 0.794), ("'ll", 0.792)]
```

### Analogies: the arithmetic and the catch

Now the catch, with three made-up words as arrows. Word P is the arrow $(3, 2)$, which is $\mathbf{b}$ itself, word Q is $(2, 4)$, which is $\mathbf{c}$ itself, and word R is $(4, 3)$. The query is $(4, 5)$, with length $\sqrt{41} = 6.40$. The cosines with the query are $22 / (3.61 \times 6.40) = 0.953$ for P, $28 / (4.47 \times 6.40) = 0.978$ for Q and $31 / (5 \times 6.40) = 0.968$ for R. The nearest word is Q, one of the input words. Many analogy programmes therefore leave the input words out of the answers, and then the nearest is R. Whether that rule is applied changes the answer, and the next sections show it on real words.

In [ ]:
def analogy(a, b, c, exclude=True, k=3):       # a is to b as c is to ?
    q = V[index[b]] - V[index[a]] + V[index[c]]
    s = U @ (q / np.linalg.norm(q))            # cosine of the query with all
    if exclude:
        for w in (a, b, c):
            s[index[w]] = -9                   # the usual rule: no input word
    return nearest(s, k)

print(analogy("man", "king", "woman", exclude=False))
print(analogy("man", "king", "woman"))
print(analogy("man", "doctor", "woman", exclude=False))
print(analogy("man", "doctor", "woman"))

*The lesson shows:*

```text
[('king', 0.855), ('queen', 0.783), ('monarch', 0.693)]
[('queen', 0.783), ('monarch', 0.693), ('throne', 0.683)]
[('doctor', 0.866), ('nurse', 0.776), ('physician', 0.713)]
[('nurse', 0.776), ('physician', 0.713), ('doctors', 0.679)]
```

### Analogies that fail

The block finds the place of the expected answer among all 10,000 words after the rule, and lists the top three words returned.

In [ ]:
def rank_of(a, b, c, answer):                  # place of the right answer
    q = V[index[b]] - V[index[a]] + V[index[c]]
    s = U @ (q / np.linalg.norm(q))
    for w in (a, b, c):
        s[index[w]] = -9
    return [words[i] for i in np.argsort(-s)].index(answer) + 1

tests = [("france", "paris", "japan", "tokyo"),
         ("germany", "berlin", "australia", "canberra"),
         ("hot", "cold", "tall", "short"),
         ("write", "wrote", "go", "went"),
         ("bird", "fly", "fish", "swim")]
for a, b, c, ans in tests:
    print(f"{a}:{b} :: {c}:{ans}  place {rank_of(a, b, c, ans)}  top",
          [w for w, _ in analogy(a, b, c)])

*The lesson shows:*

```text
france:paris :: japan:tokyo  place 1  top ['tokyo', 'osaka', 'japanese']
germany:berlin :: australia:canberra  place 6  top ['melbourne', 'sydney', 'perth']
hot:cold :: tall:short  place 53  top ['feet', 'thick', 'beneath']
write:wrote :: go:went  place 8  top ['came', 'he', 'but']
bird:fly :: fish:swim  place 60  top ['sail', 'eat', 'catch']
```

## 11.6 Embeddings are lopsided

Random pairs of real embeddings do not have cosine zero, because every vector shares a common offset. You measure it on the course’s table, remove it, and see what the papers on other embeddings found.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/lopsided-embeddings)

### Measure it on the table

The block loads the 10,000 word vectors and builds 20,000 random pairs of words, the same pairs every time they are used below. `unit` scales every row to length 1, so a dot product of two rows is their cosine.

In [ ]:
import numpy as np
import urllib.request

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
V = np.load("glove.npz")["vectors"].astype(float)   # 10,000 words by 100
unit = lambda A: A / np.linalg.norm(A, axis=1, keepdims=True)

Now the pairs, the cosine of each, and three tables to compare: the raw vectors, random vectors of the same size, and the vectors with the mean vector (the average of all 10,000, lesson 10.7) subtracted from every one.

In [ ]:
i = rng.integers(0, 10_000, 20_000)            # 20,000 random pairs of words
j = rng.integers(0, 10_000, 20_000)
i, j = i[i != j], j[i != j]

def pair_cos(M):                               # cosine of each of those pairs
    M = unit(M)
    return (M[i] * M[j]).sum(axis=1)

def show(name, c):
    print(f"{name:10s} mean {c.mean():+.4f}  spread {c.std():.4f}")

show("raw", pair_cos(V))
show("random", pair_cos(rng.normal(size=(10_000, 100))))
mean = V.mean(axis=0)
C = V - mean                                   # subtract the mean vector
show("centred", pair_cos(C))

*The lesson shows:*

```text
raw        mean +0.1753  spread 0.1388
random     mean -0.0002  spread 0.0995
centred    mean -0.0007  spread 0.1315
```

Why 0.175? With an offset $\mathbf{m}$ in every vector and independent own parts, the average dot product of two words is the squared length of $\mathbf{m}$ (multiply out $(\mathbf{m} + \mathbf{z}_a) \cdot (\mathbf{m} + \mathbf{z}_b)$: the terms holding an own part average to 0, and $\mathbf{m} \cdot \mathbf{m}$ is left), and the average of the product of two lengths is about the average squared length. So the mean cosine is about the squared length of the mean over the average squared length. The block checks that, and prints the two lengths.

In [ ]:
print(np.linalg.norm(mean).round(2), np.linalg.norm(V, axis=1).mean().round(2))
print((mean @ mean / (V ** 2).sum(axis=1).mean()).round(3))

*The lesson shows:*

```text
2.29 5.5
0.171
```

### The top directions

The **singular value decomposition** of lesson 10.5 of the centred table gives its directions in order of how much of the variance each holds. The block prints the variance share of the first five and the first eight together, then removes the top 1, 2 and 3 directions from every vector and measures the pair cosines again.

In [ ]:
_, s, Vt = np.linalg.svd(C, full_matrices=False)   # lesson 10.5
share = s ** 2 / (s ** 2).sum()                # variance share of each direction
print(share[:5].round(3), share[:8].sum().round(3))
for k in (1, 2, 3):
    R = C - (C @ Vt[:k].T) @ Vt[:k]            # take the top k directions out
    show(f"top {k} out", pair_cos(R))

*The lesson shows:*

```text
[0.052 0.042 0.038 0.032 0.03 ] 0.266
top 1 out  mean -0.0007  spread 0.1272
top 2 out  mean -0.0006  spread 0.1247
top 3 out  mean -0.0007  spread 0.1221
```

The first direction holds 5.2% of the variance, the first five hold about 19%, the first eight 26.6%, against 8% if all directions were equal. Removing the top 1, 2 and 3 directions lowers the spread only from 0.1315 to 0.1272, 0.1247 and 0.1221. So in this table the main lopsidedness is the mean vector, and the top directions are a small further effect. The words at the ends of the top direction are a clue to what it holds.

In [ ]:
words = [str(w) for w in np.load("glove.npz")["words"]]
along = C @ Vt[0]                              # position along direction 1
print([words[i] for i in np.argsort(-along)[:5]])
print([words[i] for i in np.argsort(along)[:5]])

*The lesson shows:*

```text
['bobby', 'sr.', 'eddie', 'kenny', 'footballer']
['measures', 'countries', 'reduce', 'mechanisms', 'increase']
```

One end is first names and sports, the other abstract policy words, which suggests that this direction separates kinds of text, though that is a reading of five words at each end, not a test. Does centring change which words are neighbours? The last block takes 300 random words, finds each word’s 10 nearest by cosine in the raw and in the centred table, and counts the share they have in common.

In [ ]:
queries = rng.choice(10_000, 300, replace=False)
def top10(M):
    S = unit(M)[queries] @ unit(M).T
    S[np.arange(300), queries] = -9
    return np.argsort(-S, axis=1)[:, :10]
a, b = top10(V), top10(C)
print(np.mean([len(set(x) & set(y)) / 10 for x, y in zip(a, b)]).round(3))

*The lesson shows:*

```text
0.817
```

## 11.7 Squeezing vectors: random projections and truncation

A random matrix to fewer dimensions keeps distances roughly, yet a table’s nearest neighbours are more fragile than its distances. You count what you save and what you lose, and see how some models are trained so that cutting the tail is safe.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/squeezing-vectors)

### A random matrix keeps distances roughly

Worked for the 10,000 words with $\epsilon = 0.2$: $\ln 10{,}000 = 9.21$, so $k > 8 \times 9.21 / 0.04 = 1{,}842$. That is more than the 100 numbers the vectors have, so for these words the guarantee says nothing at 50. At $k = 50$ it would hold only for $\epsilon > 1.2$, which is no guarantee. The lemma is a worst-case promise for every pair. Typical pairs do better, as the next block measures. It loads the words and sets up the three methods.

In [ ]:
import numpy as np
import urllib.request

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
V = np.load("glove.npz")["vectors"].astype(float)   # 10,000 words by 100
C = V - V.mean(axis=0)                         # centred, as in lesson 11.6
_, _, Vt = np.linalg.svd(C, full_matrices=False)    # directions of the cloud
unit = lambda A: A / np.linalg.norm(A, axis=1, keepdims=True)
print(V.shape, Vt.shape)

*The lesson shows:*

```text
(10000, 100) (100, 100)
```

The rows of `Vt` are the directions of the centred table in order of variance (lesson 10.5). Now a random 100-to-50 matrix, 5,000 random pairs of words and the ratio of each pair’s distance after to before. The code then prints the Johnson–Lindenstrauss number for 10,000 words and $\epsilon = 0.2$.

In [ ]:
M = rng.normal(size=(100, 50)) / np.sqrt(50)   # a random 100-to-50 matrix
pairs = rng.integers(0, 10_000, (5000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
P = V @ M                                      # every word, squeezed to 50
before = np.linalg.norm(V[pairs[:, 0]] - V[pairs[:, 1]], axis=1)
after = np.linalg.norm(P[pairs[:, 0]] - P[pairs[:, 1]], axis=1)
ratio = after / before
print(ratio.mean().round(3), ratio.std().round(3))
print(np.percentile(ratio, [5, 95]).round(3))
print((8 * np.log(10_000) / 0.2 ** 2).round(0))

*The lesson shows:*

```text
0.992 0.097
[0.835 1.156]
1842.0
```

### What the neighbours lose

Distances being kept roughly does not mean neighbours are kept. The first block below defines the neighbours and the recall. The second scores the three methods at $k$ = 50, 25 and 10, and prints the cut, the best directions after centring, the best directions without centring (the raw vectors projected) and a random matrix, in that order.

In [ ]:
queries = rng.choice(10_000, 300, replace=False)

def top10(M):                                  # the 10 nearest words by cosine
    S = unit(M)[queries] @ unit(M).T
    S[np.arange(300), queries] = -9            # leave the word itself out
    return np.argsort(-S, axis=1)[:, :10]

base = top10(V)
def recall(M):                                 # share of the same 10 kept
    return np.mean([len(set(a) & set(b)) / 10 for a, b in zip(base, top10(M))])

In [ ]:
for k in (50, 25, 10):
    cut = recall(V[:, :k])                     # keep the first k numbers
    pca = recall(C @ Vt[:k].T)                 # keep the k best directions
    raw = recall(V @ Vt[:k].T)                 # the same, without centring
    rand = recall(V @ (rng.normal(size=(100, k)) / np.sqrt(k)))
    print(k, round(cut, 3), round(pca, 3), round(raw, 3), round(rand, 3))

*The lesson shows:*

```text
50 0.58 0.655 0.696 0.465
25 0.345 0.455 0.487 0.327
10 0.125 0.223 0.242 0.121
```

The random matrix’s share depends on which matrix you draw. Five other matrices of the same size give the range printed by this block, from the smallest to the largest of the five at each size.

In [ ]:
for k in (50, 25, 10):                         # five other random matrices
    r = [recall(V @ (np.random.default_rng(s).normal(size=(100, k))
                     / np.sqrt(k))) for s in range(1, 6)]
    print(k, round(min(r), 3), round(max(r), 3))

*The lesson shows:*

```text
50 0.476 0.495
25 0.303 0.321
10 0.1 0.119
```

### Why neighbours are more fragile than distances

Neighbours sit close together, so a small error can reorder them. The block measures two things: the average gap in cosine between a word’s 10th and 11th nearest word, and how much a random pair’s cosine changes under the 50-number random matrix `M` from above.

In [ ]:
S = unit(V)[queries] @ unit(V).T               # cosine to all words
S[np.arange(300), queries] = -9
top = -np.sort(-S, axis=1)
print((top[:, 9] - top[:, 10]).mean().round(4))
cos_before = (unit(V)[pairs[:, 0]] * unit(V)[pairs[:, 1]]).sum(axis=1)
cos_after = (unit(P)[pairs[:, 0]] * unit(P)[pairs[:, 1]]).sum(axis=1)
print((cos_after - cos_before).std().round(3))

*The lesson shows:*

```text
0.0055
0.127
```

## 11.8 Directions as concepts

This optional lesson looks at the idea that directions in a model stand for concepts, and runs a toy model of superposition. It separates what was measured, in what kind of model, from what is still a hypothesis.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/directions-as-concepts)

### A toy you can train

The gradient is the chain rule of Module 05 applied once, and the code is given; the block after it checks one slope against a numeric one, as in lesson 04.9.

In [ ]:
import numpy as np

n, m = 5, 2                                    # 5 features, 2 directions
imp = 0.7 ** np.arange(n)                      # feature 0 matters most

def loss_grads(W, b, x):
    h = x @ W.T                                # squeeze 5 features into 2
    z = h @ W + b
    r = np.maximum(z, 0) - x                   # output minus input
    g = 2 * imp * r * (z > 0) / len(x)         # slope of the loss at z
    return (imp * r ** 2).sum() / len(x), h.T @ g + W @ (g.T @ x), g.sum(axis=0)

In [ ]:
rng = np.random.default_rng(1)                 # check the slopes with numbers
W, b = rng.normal(size=(m, n)), rng.normal(size=n) * 0.1
x = rng.random((8, n)) * (rng.random((8, n)) < 0.5)
e = np.zeros_like(W); e[1, 2] = 1e-6
slope = loss_grads(W, b, x)[1][1, 2]
numeric = (loss_grads(W + e, b, x)[0] - loss_grads(W - e, b, x)[0]) / 2e-6
print(round(slope, 6), round(numeric, 6))

*The lesson shows:*

```text
0.033297 0.033297
```

The two agree, so the gradient is right. Now plain gradient descent with a fresh batch of 256 inputs at each of 20,000 steps. Each feature is active with chance `density` (the **sparsity** is one minus that), and the loop trains at four densities and prints how many features have a direction longer than 0.5 (stored) and the angles of those directions in degrees. This is one run, with one seed.

In [ ]:
def train(density, steps=20_000, lr=0.2):
    rng = np.random.default_rng(0)
    W, b = rng.normal(size=(m, n)) * 0.5, np.zeros(n)
    for t in range(steps):
        x = rng.random((256, n)) * (rng.random((256, n)) < density)
        _, gW, gb = loss_grads(W, b, x)
        W -= lr * gW; b -= lr * gb
    return W, b

for density in (1.0, 0.5, 0.1, 0.02):
    W, b = train(density)
    kept = np.linalg.norm(W, axis=0) > 0.5     # features the model stores
    angle = np.degrees(np.arctan2(W[1], W[0]))[kept].round(0)
    print(density, kept.sum(), angle)

*The lesson shows:*

```text
1.0 2 [ 32. 122.]
0.5 3 [ 24. 114. -66.]
0.1 4 [  69.  159.  -21. -111.]
0.02 5 [  67.  142.   -9.  -79. -146.]
```

## 11.9 Project: audit an embedding table

You audit the course’s word vectors: the cosine of random pairs, the effect of centring and how much cutting the tail hurts neighbour recall. It turns the module’s measuring habits into a short report.

[Read the lesson](https://aicourses.dev/mathematics/high-dimensions/project-embedding-audit)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: load the table

The same block as in the lessons before: it loads the vectors and defines `unit`, which scales every row to length 1.

In [ ]:
import numpy as np
import urllib.request

rng = np.random.default_rng(0)
DATA = "https://raw.githubusercontent.com/aicoursesdev/mathematics-for-ai/main/data/"
urllib.request.urlretrieve(DATA + "glove-10k-100d.npz", "glove.npz")
V = np.load("glove.npz")["vectors"].astype(float)   # 10,000 words by 100
unit = lambda A: A / np.linalg.norm(A, axis=1, keepdims=True)
print(V.shape)

*The lesson shows:*

```text
(10000, 100)
```

### Step 2: random pairs before and after centring

Pick 20,000 random pairs of words, once, and measure each pair’s cosine in the raw table and in the centred one. Use the same pairs for both, so that the only difference is the centring. The block prints the mean, the spread and the 5th and 95th percentiles: the range random pairs reach.

In [ ]:
i = rng.integers(0, 10_000, 20_000)            # 20,000 random pairs of words
j = rng.integers(0, 10_000, 20_000)
i, j = i[i != j], j[i != j]

def pair_cos(M):                               # cosine of each of those pairs
    M = unit(M)
    return (M[i] * M[j]).sum(axis=1)

C = V - V.mean(axis=0)                         # subtract the mean vector
for name, M in (("raw", V), ("centred", C)):
    c = pair_cos(M)
    print(name, c.mean().round(4), c.std().round(4),
          np.percentile(c, [5, 95]).round(3))

*The lesson shows:*

```text
raw 0.1753 0.1388 [-0.029  0.42 ]
centred -0.0007 0.1315 [-0.191  0.236]
```

### Step 3: cutting the tail

Take 300 random words and find each one’s 10 nearest words by cosine in the full table. Then cut the vectors to $k$ numbers and see how many of the 10 are still found. The last column projects the raw vectors onto the best directions without centring first, so its cosines are not changed by the centring.

In [ ]:
queries = rng.choice(10_000, 300, replace=False)

def top10(M):                                  # 10 nearest words by cosine
    S = unit(M)[queries] @ unit(M).T
    S[np.arange(300), queries] = -9
    return np.argsort(-S, axis=1)[:, :10]

base = top10(V)
def recall(M):                                 # share of the same 10 kept
    return np.mean([len(set(a) & set(b)) / 10 for a, b in zip(base, top10(M))])

_, _, Vt = np.linalg.svd(C, full_matrices=False)
print("keep  first-k  best-k-centred  best-k-raw")
for k in (90, 75, 50, 25, 10):
    print(f"{k:4d}  {recall(V[:, :k]):.3f}    {recall(C @ Vt[:k].T):.3f}"
          f"           {recall(V @ Vt[:k].T):.3f}")

*The lesson shows:*

```text
keep  first-k  best-k-centred  best-k-raw
  90  0.869    0.812           0.950
  75  0.767    0.784           0.867
  50  0.588    0.672           0.704
  25  0.363    0.473           0.497
  10  0.130    0.227           0.239
```